# Custom TLM Tokenizer

Building the vocabulary

In [25]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.tokenizer import TLM_Tokenizer

In [26]:
tokenizer = TLM_Tokenizer(vocab_size=10000)
test_sample = "def add(a, b):\n    '''Add two numbers'''\n    return a + b\n"
tokenizer.train([test_sample])

pretokenized = tokenizer.pretokenize(tokenizer.normalize_code(test_sample))
print("pretokenized tokens:", pretokenized)
print()

ids = tokenizer.encode(test_sample)
print("encoded:", ids)
print()

decoded = tokenizer.decode(ids, debug=True)
print("decoded:\n", decoded)

Trained code tokenizer with vocab_size=10000
Special token verification:
  pad_token: '<pad>' → id 0 → '<pad>'
  unk_token: '<unk>' → id 1 → '<unk>'
  bos_token: '<bos>' → id 2 → '<bos>'
  eos_token: '<eos>' → id 3 → '<eos>'
  mask_token: '<mask>' → id 4 → '<mask>'
  sep_token: '<sep>' → id 5 → '<sep>'
  indent_token: '<indent>' → id 6 → '<indent>'
  dedent_token: '<dedent>' → id 7 → '<dedent>'
  newline_token: '<newline>' → id 8 → '<newline>'
  rename_token: '<rename>' → id 9 → '<rename>'
  simplify_token: '<simplify>' → id 10 → '<simplify>'
  deduplicate_token: '<deduplicate>' → id 11 → '<deduplicate>'
  remove_dead_code_token: '<remove_dead_code>' → id 12 → '<remove_dead_code>'
  extract_method_token: '<extract_method>' → id 13 → '<extract_method>'
  diff_start: '<<<<' → id 14 → '<<<<'
  diff_sep: '====' → id 15 → '===='
  diff_end: '>>>>' → id 16 → '>>>>'
  search_token: '<search>' → id 17 → '<search>'
  replace_token: '<replace>' → id 18 → '<replace>'
  extra_id_0: '<extra_id_0>' 

## Training

In [27]:
import pandas as pd

corpus = pd.read_csv("../data/code_search_net.csv")
codes = corpus["code"].tolist()

In [28]:
tokenizer = TLM_Tokenizer()
tokenizer.train(codes)

Trained code tokenizer with vocab_size=10000
Special token verification:
  pad_token: '<pad>' → id 0 → '<pad>'
  unk_token: '<unk>' → id 1 → '<unk>'
  bos_token: '<bos>' → id 2 → '<bos>'
  eos_token: '<eos>' → id 3 → '<eos>'
  mask_token: '<mask>' → id 4 → '<mask>'
  sep_token: '<sep>' → id 5 → '<sep>'
  indent_token: '<indent>' → id 6 → '<indent>'
  dedent_token: '<dedent>' → id 7 → '<dedent>'
  newline_token: '<newline>' → id 8 → '<newline>'
  rename_token: '<rename>' → id 9 → '<rename>'
  simplify_token: '<simplify>' → id 10 → '<simplify>'
  deduplicate_token: '<deduplicate>' → id 11 → '<deduplicate>'
  remove_dead_code_token: '<remove_dead_code>' → id 12 → '<remove_dead_code>'
  extract_method_token: '<extract_method>' → id 13 → '<extract_method>'
  diff_start: '<<<<' → id 14 → '<<<<'
  diff_sep: '====' → id 15 → '===='
  diff_end: '>>>>' → id 16 → '>>>>'
  search_token: '<search>' → id 17 → '<search>'
  replace_token: '<replace>' → id 18 → '<replace>'
  extra_id_0: '<extra_id_0>' 

In [29]:
pretokenized = tokenizer.pretokenize(tokenizer.normalize_code(codes[15]))
print("pretokenized tokens:", pretokenized)
print()

ids = tokenizer.encode(codes[15])
print("encoded:", list(ids))
print()

decoded = tokenizer.decode(ids, debug=True)
print("decoded:\n", decoded)

pretokenized tokens: ['def', '_infer_schema', '(', 'row', ',', 'names', '=', 'None', ')', ':', '<newline>', '<indent>', '"""Infer the schema from dict/namedtuple/object"""', '<newline>', 'if', 'isinstance', '(', 'row', ',', 'dict', ')', ':', '<newline>', '<indent>', 'items', '=', 'sorted', '(', 'row', '.', 'items', '(', ')', ')', '<newline>', '<newline>', '<dedent>', 'elif', 'isinstance', '(', 'row', ',', '(', 'tuple', ',', 'list', ')', ')', ':', '<newline>', '<indent>', 'if', 'hasattr', '(', 'row', ',', '"__fields__"', ')', ':', '# Row', '<newline>', '<indent>', 'items', '=', 'zip', '(', 'row', '.', '__fields__', ',', 'tuple', '(', 'row', ')', ')', '<newline>', '<dedent>', 'elif', 'hasattr', '(', 'row', ',', '"_fields"', ')', ':', '# namedtuple', '<newline>', '<indent>', 'items', '=', 'zip', '(', 'row', '.', '_fields', ',', 'tuple', '(', 'row', ')', ')', '<newline>', '<dedent>', 'else', ':', '<newline>', '<indent>', 'if', 'names', 'is', 'None', ':', '<newline>', '<indent>', 'names', '

In [30]:
tokenizer.save("../src/tokenizer/tokenizer_tlm_small_hybrid.json")

## Validation -- full-corpus round-trip check


In [31]:
# Manual encode/decode inspection
sample = """def add(a, b):
    '''Add two numbers'''
    return a + b
"""

norm = tokenizer.normalize_code(sample)
pretokens = tokenizer.pretokenize(norm)
ids = tokenizer.encode(sample)
tokens_from_ids = [tokenizer.id_to_token(i) for i in ids]
decoded = tokenizer.decode(ids, debug=True)

print("=== normalized ===")
print(repr(norm))

print("\n=== pretokenized (pre-BPE) ===")
print(pretokens)

print("\n=== encoded ids ===")
print(ids)

print("\n=== ids -> BPE tokens ===")
print(tokens_from_ids)

print("\n=== decoded (debug=True, shows special tokens) ===")
print(decoded)

print("\n=== decoded (debug=False, clean output) ===")
print(tokenizer.decode(ids, debug=False))

=== normalized ===
"def add(a, b):\n    '''Add two numbers'''\n    return a + b\n"

=== pretokenized (pre-BPE) ===
['def', 'add', '(', 'a', ',', 'b', ')', ':', '<newline>', '<indent>', "'''Add two numbers'''", '<newline>', 'return', 'a', '+', 'b', '<newline>', '<dedent>']

=== encoded ids ===
[6040, 6816, 4531, 4550, 4542, 4588, 4543, 4539, 8, 6, 7653, 7732, 7818, 8295, 176, 6375, 8, 6028, 4550, 4623, 4588, 8, 7]

=== ids -> BPE tokens ===
['def</w>', 'add</w>', '(</w>', 'a</w>', ',</w>', 'b</w>', ')</w>', ':</w>', '<newline>', '<indent>', "'''", 'Add</w>', 'two</w>', 'number', 's', "'''</w>", '<newline>', 'return</w>', 'a</w>', '+</w>', 'b</w>', '<newline>', '<dedent>']

=== decoded (debug=True, shows special tokens) ===
def add(a,b):
    [<newline>]'''Add two numbers'''
    [<newline>]return a+b
[<newline>]

=== decoded (debug=False, clean output) ===
def add(a,b):
    '''Add two numbers'''
    return a+b


In [32]:
import json
import ast
import tokenize

total = 0
bad = 0
fallback_count = 0
examples = []

for t in range(1, 6):
    with open(f"../data/curriculum/tier{t}.jsonl", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            code = rec["code"]
            total += 1

            norm = tokenizer.normalize_code(code)
            try:
                tokenizer._pretokenize_stdlib(norm)
            except (tokenize.TokenError, IndentationError, SyntaxError, ValueError):
                fallback_count += 1

            ids = tokenizer.encode(code)
            dec = tokenizer.decode(ids)
            try:
                ast.parse(dec)
            except SyntaxError:
                bad += 1
                if len(examples) < 10:
                    examples.append((code, dec))

rate = 100 * bad / total if total else 0.0
fb_rate = 100 * fallback_count / total if total else 0.0
print(f"{bad}/{total} decode-invalid ({rate:.3f}%)")
print(f"lenient fallback triggered: {fallback_count}/{total} ({fb_rate:.3f}%)")
print()

for orig, dec in examples:
    print("=== mismatch ===")
    print("ORIG:", orig[:300])
    print("DEC: ", dec[:300])
    print()

<unknown>:68: SyntaxWarning: invalid octal escape sequence '\664'


2/194627 decode-invalid (0.001%)
lenient fallback triggered: 0/194627 (0.000%)

=== mismatch ===
ORIG: def do_actions(self, target_filename, dry_run=False):
        """
        Runs all the given action rules in this ruleset on target_filename
        :param target_filename:
        :retrn: filename Filename and path after any actions have been completed
\        """
        for rule in self.action_r
DEC:  def do_actions(self,target_filename,dry_run=False):
    """
    Runs all the given action rules in this ruleset on target_filename
    :param target_filename:
    :retrn: filename Filename and path after any actions have been completed
    \"""
    for rule in self.action_rules:
        target_filen

=== mismatch ===
ORIG: def show(self, temp_file_name = 'ani.mp4', **kwargs):
        """

        ## Arguments:
        - 'args' and 'kwargs' will be passed to 'self.save()'
        """
        ## [NOTE] Make this method as a method of base class.

        ## [NOTE] This should be modifi